In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
import subprocess
import sys

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/shoe_project")
bundles = sorted(PROJECT.glob("shoe_absa_*.zip"))
if not bundles:
    raise FileNotFoundError(f"{PROJECT} 안에 shoe_absa_*.zip 파일이 없습니다.")

bundle = bundles[-1]
ROOT = Path("/content") / bundle.stem
ROOT.mkdir(exist_ok=True)

print("사용할 ZIP:", bundle.name)
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(ROOT)

print((ROOT / "requirements.txt").read_text())
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-r", str(ROOT / "requirements.txt")],
    check=True,
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
사용할 ZIP: shoe_absa_*.zip
sentence-transformers==6.3.0
transformers==5.23.0
tokenizers==0.23.3
nltk==3.9.1


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', '/content/shoe_absa_*/requirements.txt'], returncode=0)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import nltk

from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 연결되지 않았습니다. 런타임 유형에서 GPU를 선택하세요.")

device = torch.device("cuda")
print("사용 GPU:", torch.cuda.get_device_name(0))

settings = json.loads((ROOT / "settings.json").read_text(encoding="utf-8"))

def read_frame(name):
    # "None"으로 이름 붙인 군집과 리뷰 원문의 문자열을 그대로 읽는다.
    return pd.read_csv(ROOT / f"{name}.csv.gz", keep_default_na=False)

shoe_train_df = read_frame("train")
shoe_test_df = read_frame("test")
check_reviews = read_frame("check_reviews")
check_df = read_frame("check_df")
check_passed = read_frame("check_passed")

with np.load(ROOT / "clusters.npz") as arrays:
    cluster_ids = arrays["cluster_ids"]
    cluster_centroids = arrays["cluster_centroids"]

cluster_to_aspect = {int(k): v for k, v in settings["cluster_to_aspect"].items()}
COSINE_THRESHOLD = settings["threshold"]
ASPECT_EN = settings["aspect_en"]
SENTIMENTS = settings["sentiments"]

assert len(shoe_train_df) == settings["train_count"]
assert len(shoe_test_df) == settings["test_count"]
assert len(check_passed) == settings["check_passed_count"]
assert set(cluster_ids) == set(cluster_to_aspect)
assert cluster_centroids.shape == (len(cluster_ids), 768)

sbert = SentenceTransformer(str(ROOT / "sbert"), device="cuda")
tokenizer = AutoTokenizer.from_pretrained(ROOT / "absa", local_files_only=True)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    ROOT / "absa", local_files_only=True
).to(device).eval()

id_to_label = {
    int(k): v.lower() for k, v in sentiment_model.config.id2label.items()
}
assert set(id_to_label.values()) == set(SENTIMENTS)

for resource in ["punkt", "punkt_tab"]:
    nltk.download(resource, quiet=True, raise_on_error=True)

# 같은 ZIP으로 다시 접속하면 같은 결과 폴더를 사용한다.
OUT = PROJECT / f"results_{settings['bundle_id']}"
OUT.mkdir(exist_ok=True)

print("Train / Test:", len(shoe_train_df), "/", len(shoe_test_df))
print("대표벡터:", cluster_centroids.shape, "| 임계값:", COSINE_THRESHOLD)
print("이전 감성 판정 결과:", len(check_passed), "문장")
print("결과 저장 위치:", OUT)

사용 GPU: NVIDIA L4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Train / Test: 76320 / 19080
대표벡터: (30, 768) | 임계값: 0.6
이전 감성 판정 결과: 82 문장
결과 저장 위치: /content/drive/MyDrive/shoe_project/results_shoe_absa_20261123_094430


In [ ]:
@torch.inference_mode()
def predict_sentiments(frame, batch_size=32):
    labels = []

    for start in range(0, len(frame), batch_size):
        batch = frame.iloc[start:start + batch_size]
        targets = [ASPECT_EN[a] for a in batch["aspect_candidate"]]

        inputs = tokenizer(
            batch["sentence"].tolist(),
            text_pair=targets,
            padding=True,
            truncation="only_first",
            max_length=256,
            return_tensors="pt",
        ).to(device)

        ids = sentiment_model(**inputs).logits.argmax(dim=1).cpu().tolist()
        labels.extend(id_to_label[i] for i in ids)

    return labels


sentence_columns = [
    "review_id", "sentence_no", "sentence", "nearest_cluster",
    "cosine_similarity", "aspect_candidate", "sentiment",
]

def process_reviews(reviews):
    rows = []
    for review in reviews.itertuples(index=False):
        for no, sentence in enumerate(
            sent_tokenize(review.text, language="english"), start=1
        ):
            sentence = sentence.strip()
            if 3 <= len(sentence.split()) <= 60:
                rows.append((review.review_id, no, sentence))

    if not rows:
        return pd.DataFrame(columns=sentence_columns)

    frame = pd.DataFrame(rows, columns=["review_id", "sentence_no", "sentence"])
    vectors = sbert.encode(
        frame["sentence"].tolist(),
        batch_size=64,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    similarities = vectors @ cluster_centroids.T
    nearest = similarities.argmax(axis=1)

    frame["nearest_cluster"] = cluster_ids[nearest]
    frame["cosine_similarity"] = similarities[np.arange(len(frame)), nearest]
    frame["aspect_candidate"] = frame["nearest_cluster"].map(cluster_to_aspect)

    keep = (
        frame["cosine_similarity"].ge(COSINE_THRESHOLD)
        & frame["aspect_candidate"].notna()
        & frame["aspect_candidate"].ne("None")
    )
    passed = frame.loc[keep].copy()
    passed["sentiment"] = predict_sentiments(passed)
    return passed[sentence_columns]


gpu_labels = predict_sentiments(check_passed)
comparison = check_passed[["sentence", "aspect_candidate", "sentiment"]].copy()
comparison["colab_sentiment"] = gpu_labels
matches = comparison["sentiment"].eq(comparison["colab_sentiment"])

print(f"로컬과 Colab의 감성 판정 일치: {matches.sum()} / {len(matches)}")
if not matches.all():
    display(comparison.loc[~matches])

로컬과 Colab의 감성 판정 일치: 82 / 82


In [ ]:
from tqdm.auto import tqdm

REVIEWS_PER_CHUNK = 500

def run_split(reviews, split):
    folder = OUT / f"{split}_chunks_{REVIEWS_PER_CHUNK}"
    folder.mkdir(exist_ok=True)
    paths = []

    starts = range(0, len(reviews), REVIEWS_PER_CHUNK)
    for start in tqdm(starts, desc=split):
        stop = min(start + REVIEWS_PER_CHUNK, len(reviews))
        path = folder / f"{start:06d}_{stop:06d}.parquet"
        paths.append(path)

        if path.exists():
            continue

        result = process_reviews(reviews.iloc[start:stop])
        temporary = path.with_suffix(".partial")
        result.to_parquet(temporary, index=False, engine="pyarrow")
        temporary.replace(path)

    return pd.concat(
        [pd.read_parquet(path) for path in paths],
        ignore_index=True,
    )

train_sentence_df = run_split(shoe_train_df, "train")
test_sentence_df = run_split(shoe_test_df, "test")

print("Train에서 채택한 문장:", len(train_sentence_df))
print("Test에서 채택한 문장:", len(test_sentence_df))
display(train_sentence_df.head())

train:   0%|          | 0/153 [00:00<?, ?it/s]

test:   0%|          | 0/39 [00:00<?, ?it/s]

Train에서 채택한 문장: 60513
Test에서 채택한 문장: 14803


,review_id,sentence_no,sentence,nearest_cluster,cosine_similarity,aspect_candidate,sentiment
0,0,3,His feet are a little narrow and these are pre...,18,0.610517,사이즈/핏,negative
1,2,2,My 5 year old said they are comfy and she love...,7,0.697625,착화감,positive
2,3,2,"They fit nice, a little difficult for her to f...",23,0.712251,사이즈/핏,positive
3,4,3,I wear them working landscaping every day and ...,7,0.629408,착화감,positive
4,4,4,They are super durable and super comfortable!,7,0.717494,착화감,positive


In [ ]:
feature_columns = [
    f"{aspect}_{sentiment}"
    for aspect in ASPECT_EN
    for sentiment in SENTIMENTS
]

def make_review_vectors(reviews, sentence_results):
    review_ids = pd.Index(reviews["review_id"], name="review_id")
    assert review_ids.is_unique

    if sentence_results.empty:
        X = pd.DataFrame(0, index=review_ids, columns=feature_columns)
    else:
        feature = (
            sentence_results["aspect_candidate"] + "_" + sentence_results["sentiment"]
        )
        counts = pd.crosstab(sentence_results["review_id"], feature)
        X = counts.reindex(
            index=review_ids, columns=feature_columns, fill_value=0
        ).gt(0)

    X = X.astype("uint8")
    X.columns.name = None
    y = reviews.set_index("review_id")["rating"].reindex(X.index)
    assert y.between(1, 5).all()
    return X, y

X_train, y_train = make_review_vectors(shoe_train_df, train_sentence_df)
X_test, y_test = make_review_vectors(shoe_test_df, test_sentence_df)

for name, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
    assert X.index.equals(y.index)
    assert X.shape[1] == 27
    X.join(y).to_parquet(OUT / f"{name}_review_vectors.parquet")
    covered = X.any(axis=1)
    print(f"{name}: {X.shape}, aspect 추출 리뷰 {covered.sum():,}개 ({covered.mean():.1%})")

train: (76320, 27), aspect 추출 리뷰 39,554개 (51.8%)
test: (19080, 27), aspect 추출 리뷰 9,716개 (50.9%)


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# 새 세션에서도 셀 1~2로 OUT 경로를 복원한 후 여기부터 실행할 수 있다.
train_vectors = pd.read_parquet(OUT / "train_review_vectors.parquet")
test_vectors = pd.read_parquet(OUT / "test_review_vectors.parquet")

X_train = train_vectors.drop(columns="rating")
y_train = train_vectors["rating"]
X_test = test_vectors[X_train.columns]
y_test = test_vectors["rating"]

models = {
    "Train median": DummyRegressor(strategy="median"),
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(
        n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
}

has_pos = X_test.filter(regex="_positive$").any(axis=1)
has_neg = X_test.filter(regex="_negative$").any(axis=1)

predictions = pd.DataFrame({"actual_rating": y_test})
predictions["group"] = np.select(
    [has_pos & has_neg, has_pos | has_neg],
    ["mixed", "single"],
    default="neutral_or_unassigned",
)

metrics = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predicted = np.clip(model.predict(X_test), 1, 5)
    predictions[model_name] = predicted
    absolute_error = predictions["actual_rating"].sub(predicted).abs()

    for group in ["all", "mixed", "single", "neutral_or_unassigned"]:
        errors = (
            absolute_error if group == "all"
            else absolute_error[predictions["group"].eq(group)]
        )
        metrics.append({
            "model": model_name,
            "group": group,
            "n_reviews": len(errors),
            "MAE": errors.mean(),
        })

metrics = pd.DataFrame(metrics)
rating_distribution = pd.crosstab(
    predictions["group"], predictions["actual_rating"]
).reindex(columns=[1, 2, 3, 4, 5], fill_value=0)

display(metrics.pivot(index="model", columns="group", values="MAE").round(3))
display(rating_distribution)

predictions.to_parquet(OUT / "test_predictions.parquet")
metrics.to_csv(OUT / "experiment1_mae.csv", index=False)
rating_distribution.to_csv(OUT / "test_group_rating_counts.csv")
print("실험 1 결과 폴더:", OUT)

group,all,mixed,neutral_or_unassigned,single
model,,,,
Linear regression,0.958,1.075,1.096,0.780
Random forest,0.938,1.049,1.079,0.757
Train median,1.024,1.676,0.885,1.102


actual_rating,1,2,3,4,5
group,,,,,
mixed,164,150,235,257,288
neutral_or_unassigned,1024,520,811,1329,6040
single,1048,651,838,1284,4441


실험 1 결과 폴더: /content/drive/MyDrive/shoe_project/results_shoe_absa_20261123_094430


In [ ]:
error = (predictions["actual_rating"] - predictions["Random forest"]).abs()
print(f"1점 이내: {(error <= 1).mean():.1%}")
print(f"1점 초과: {(error > 1).mean():.1%}")

1점 이내: 75.7%
1점 초과: 24.3%


### 기존 리뷰 벡터와 예측결과 불러오기

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/shoe_project")
RUN_ID = "shoe_absa_20261123_094430"
RESULT_DIR = PROJECT_DIR / f"results_{RUN_ID}"
ANALYSIS_DIR = RESULT_DIR / "experiment1_followup"

required_files = [
    "train_review_vectors.parquet",
    "test_review_vectors.parquet",
    "test_predictions.parquet",
]
for name in required_files:
    if not (RESULT_DIR / name).is_file():
        raise FileNotFoundError(RESULT_DIR / name)

ANALYSIS_DIR.mkdir(exist_ok=True)

train_v = pd.read_parquet(RESULT_DIR / "train_review_vectors.parquet")
test_v = pd.read_parquet(RESULT_DIR / "test_review_vectors.parquet")
pred = pd.read_parquet(RESULT_DIR / "test_predictions.parquet")

assert train_v.index.is_unique and test_v.index.is_unique and pred.index.is_unique
assert test_v.index.difference(pred.index).empty
assert pred.index.difference(test_v.index).empty
pred = pred.loc[test_v.index].copy()

x27_train = train_v.drop(columns="rating")
x27_test = test_v.drop(columns="rating").loc[:, x27_train.columns]
y_train_run = train_v["rating"]
y_test_run = test_v["rating"]

assert len(train_v) == 76320 and len(test_v) == 19080
assert x27_train.shape[1] == 27
assert x27_train.isin([0, 1]).all().all()
assert x27_test.isin([0, 1]).all().all()
assert np.array_equal(pred["actual_rating"], y_test_run)

polarities = ["positive", "neutral", "negative"]

def collapse_to_three(features):
    result = pd.DataFrame(index=features.index)
    for polarity in polarities:
        cols = [c for c in features if c.endswith("_" + polarity)]
        assert len(cols) == 9, (polarity, cols)
        result[polarity] = features[cols].any(axis=1).astype("uint8")
    return result

x3_train = collapse_to_three(x27_train)
x3_test = collapse_to_three(x27_test)

pos = x3_test["positive"].eq(1)
neg = x3_test["negative"].eq(1)
neu = x3_test["neutral"].eq(1)

detail_group = pd.Series(
    np.select(
        [pos & neg, pos | neg, neu],
        ["mixed", "single", "neutral_only"],
        default="unassigned",
    ),
    index=x27_test.index,
    name="detail_group",
)

# 세분한 그룹을 원래 정의로 되돌렸을 때 기존 저장 결과와 일치해야 한다.
original_group = detail_group.replace({
    "neutral_only": "neutral_or_unassigned",
    "unassigned": "neutral_or_unassigned",
})
assert original_group.eq(pred["group"]).all()

print("결과 저장 위치:", ANALYSIS_DIR)
print("학습 / 테스트 리뷰:", len(train_v), "/", len(test_v))
display(detail_group.value_counts().rename("n_reviews"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
결과 저장 위치: /content/drive/MyDrive/shoe_project/results_shoe_absa_20261123_094430/experiment1_followup
학습 / 테스트 리뷰: 76320 / 19080


,n_reviews
detail_group,
unassigned,9364
single,8262
mixed,1094
neutral_only,360


### 추가 실험 1 - 감성 3차원 vs 속성별 감성 27차원

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

models_3d = {
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),
}

rows = []
for model_name, model in models_3d.items():
    model.fit(x3_train, y_train_run)
    prediction_3d = pd.Series(
        np.clip(model.predict(x3_test), 1, 5),
        index=x3_test.index,
    )
    prediction_27d = pred[model_name]

    error_3d = (y_test_run - prediction_3d).abs()
    error_27d = (y_test_run - prediction_27d).abs()

    masks = {
        "all": pd.Series(True, index=x3_test.index),
        "covered": detail_group.ne("unassigned"),
        "mixed": detail_group.eq("mixed"),
        "single": detail_group.eq("single"),
        "neutral_only": detail_group.eq("neutral_only"),
        "unassigned": detail_group.eq("unassigned"),
    }

    for group, mask in masks.items():
        mae_3d = error_3d.loc[mask].mean()
        mae_27d = error_27d.loc[mask].mean()
        rows.append({
            "model": model_name,
            "group": group,
            "n_reviews": int(mask.sum()),
            "MAE_3D": mae_3d,
            "MAE_27D": mae_27d,
            "gain_from_27D": mae_3d - mae_27d,
        })

comparison_3d_27d = pd.DataFrame(rows)
comparison_3d_27d.to_csv(ANALYSIS_DIR / "02_3d_vs_27d.csv", index=False)
display(comparison_3d_27d.round(4))

,model,group,n_reviews,MAE_3D,MAE_27D,gain_from_27D
0,Linear regression,all,19080,0.9834,0.9578,0.0256
1,Linear regression,covered,9716,0.8800,0.8280,0.0520
2,Linear regression,mixed,1094,1.2082,1.0754,0.1328
3,Linear regression,single,8262,0.8129,0.7800,0.0329
4,Linear regression,neutral_only,360,1.4218,1.1774,0.2444
5,Linear regression,unassigned,9364,1.0907,1.0924,-0.0017
6,Random forest,all,19080,0.9850,0.9381,0.0468
7,Random forest,covered,9716,0.8951,0.8030,0.0921
8,Random forest,mixed,1094,1.1998,1.0494,0.1505
9,Random forest,single,8262,0.8294,0.7574,0.0721


### 추가실험2 - 미배정과 중립을 분리해 전체 테스트 성능 계산

In [ ]:
audit_test = pd.DataFrame({
    "detail_group": detail_group,
    "actual_rating": y_test_run,
    "RF_prediction": pred["Random forest"],
})

audit_test["RF_abs_error"] = (
    audit_test["actual_rating"] - audit_test["RF_prediction"]
).abs()
audit_test["median_abs_error"] = (
    audit_test["actual_rating"] - pred["Train median"]
).abs()

group_summary = (
    audit_test.groupby("detail_group")
    .agg(
        n_reviews=("actual_rating", "size"),
        actual_mean=("actual_rating", "mean"),
        actual_median=("actual_rating", "median"),
        RF_mean_prediction=("RF_prediction", "mean"),
        RF_MAE=("RF_abs_error", "mean"),
        baseline_MAE=("median_abs_error", "mean"),
    )
    .reindex(["mixed", "single", "neutral_only", "unassigned"])
)
group_summary["share_pct"] = 100 * group_summary["n_reviews"] / len(audit_test)
group_summary["RF_gain"] = group_summary["baseline_MAE"] - group_summary["RF_MAE"]

group_summary.to_csv(ANALYSIS_DIR / "03_test_group_summary.csv")
display(group_summary.round(4))

unassigned_predictions = audit_test.loc[
    detail_group.eq("unassigned"), "RF_prediction"
]
assert unassigned_predictions.nunique() == 1
print("미배정 리뷰에 랜덤포레스트가 공통으로 예측한 별점:",
      round(unassigned_predictions.iloc[0], 4))

,n_reviews,actual_mean,actual_median,RF_mean_prediction,RF_MAE,baseline_MAE,share_pct,RF_gain
detail_group,,,,,,,,
mixed,1094,3.3245,3.0,3.3923,1.0494,1.6755,5.7338,0.6261
single,8262,3.8980,5.0,3.8904,0.7574,1.1020,43.3019,0.3447
neutral_only,360,3.4361,4.0,3.3703,1.1030,1.5639,1.8868,0.4609
unassigned,9364,4.1410,5.0,4.1515,1.0783,0.8590,49.0776,-0.2193


미배정 리뷰에 랜덤포레스트가 공통으로 예측한 별점: 4.1515


In [ ]:
# 수정 전
bundle = bundles[-1]

# 수정 후 — 다른 ZIP이 선택되는 경우에만
bundle = PROJECT / "shoe_absa_20261123_094430.zip"

In [ ]:
from nltk.tokenize import sent_tokenize

assert settings["bundle_id"] == RUN_ID, "셀 A와 기존 셀 2의 ZIP이 다릅니다."

reviews_by_id = shoe_train_df.set_index("review_id", verify_integrity=True)
assert x27_train.index.difference(reviews_by_id.index).empty

train_covered = x27_train.any(axis=1)

def sample_ids(mask, n=50):
    return mask.loc[mask].sample(n=n, random_state=42).index

unassigned_ids = sample_ids(~train_covered)
assigned_ids = sample_ids(train_covered)

sample_ids_all = unassigned_ids.append(assigned_ids)
sample_reviews = reviews_by_id.loc[sample_ids_all, ["text"]].copy()
sample_reviews["sample_group"] = np.where(
    train_covered.loc[sample_reviews.index], "assigned", "unassigned"
)

# 1. 复원한 전체 문장을 기록한다. 길이 기준은 기존 process_reviews와 동일하다.
sentence_rows = []
for review_id, text in sample_reviews["text"].items():
    for no, sentence in enumerate(sent_tokenize(text, language="english"), start=1):
        sentence = sentence.strip()
        word_count = len(sentence.split())
        sentence_rows.append({
            "review_id": review_id,
            "sentence_no": no,
            "sentence": sentence,
            "word_count": word_count,
            "status": (
                "too_short" if word_count < 3 else
                "too_long" if word_count > 60 else
                "eligible"
            ),
        })

trace = pd.DataFrame(sentence_rows, columns=[
    "review_id", "sentence_no", "sentence", "word_count", "status"
])
trace["aspect_candidate"] = pd.Series(index=trace.index, dtype="object")
trace["cosine_similarity"] = np.nan

# 2. 길이 조건을 통과한 문장만 원래 SBERT·대표벡터·임계값으로 재판정한다.
eligible_index = trace.index[trace["status"].eq("eligible")]
if len(eligible_index):
    embeddings = sbert.encode(
        trace.loc[eligible_index, "sentence"].tolist(),
        batch_size=64,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    similarities = embeddings @ cluster_centroids.T
    nearest = similarities.argmax(axis=1)
    scores = similarities[np.arange(len(nearest)), nearest]
    aspects = np.array([cluster_to_aspect[int(k)] for k in cluster_ids[nearest]])

    below = scores < COSINE_THRESHOLD
    no_aspect = pd.isna(aspects) | (aspects == "None")

    trace.loc[eligible_index, "aspect_candidate"] = aspects
    trace.loc[eligible_index, "cosine_similarity"] = scores
    trace.loc[eligible_index, "status"] = np.select(
        [below & no_aspect, below, no_aspect],
        ["below_and_none", "below_threshold", "none_cluster"],
        default="accepted",
    )

# 3. 재판정 결과를 저장된 원래 27차원 벡터와 대조한다.
# 발견된 속성 집합 자체가 일치해야 수작업 분석에 사용한다.
accepted = trace.loc[trace["status"].eq("accepted")]
reconstructed = accepted.groupby("review_id")["aspect_candidate"].agg(set)

for review_id in sample_reviews.index:
    vector = x27_train.loc[review_id]
    saved_aspects = {col.rsplit("_", 1)[0] for col in vector.index[vector.eq(1)]}
    assert saved_aspects == reconstructed.get(review_id, set()), (
        f"review_id={review_id}: 저장된 속성과 재판정 결과가 다릅니다."
    )

# 4. 문장 단위 결과와 리뷰 단위 요약을 각각 보관한다.
trace = trace.join(sample_reviews["sample_group"], on="review_id")
trace.to_csv(
    ANALYSIS_DIR / "03_sentence_trace.csv", index=False, encoding="utf-8-sig"
)

statuses = [
    "too_short", "too_long", "below_threshold",
    "none_cluster", "below_and_none", "accepted",
]
reason_counts = pd.crosstab(trace["review_id"], trace["status"]).reindex(
    index=sample_reviews.index, columns=statuses, fill_value=0
)
review_trace = sample_reviews.join(reason_counts)
review_trace["no_sentence"] = reason_counts.sum(axis=1).eq(0)
review_trace.to_csv(ANALYSIS_DIR / "03_review_trace.csv", encoding="utf-8-sig")

print("미배정 학습 리뷰 50개에서 각 현상이 한 번 이상 나타난 리뷰 수")
unassigned_trace = review_trace.loc[review_trace["sample_group"].eq("unassigned")]
display(unassigned_trace[statuses + ["no_sentence"]].gt(0).sum().rename("n_reviews"))

# 5. 별점·모델 결과를 가린 수작업 검토표를 만든다.
# 재실행해도 이미 작성한 내용을 덮어쓰지 않는다.
manual_path = ANALYSIS_DIR / "03_manual_review.csv"
if not manual_path.exists():
    manual = sample_reviews[["text"]].sample(frac=1, random_state=42).copy()
    manual["human_aspects"] = ""
    manual["note"] = ""
    manual.to_csv(manual_path, encoding="utf-8-sig")

print("수작업 검토표:", manual_path)
print("문장별 탈락 내역:", ANALYSIS_DIR / "03_sentence_trace.csv")
print("입력 가능한 속성:", list(settings["aspect_en"]))

미배정 학습 리뷰 50개에서 각 현상이 한 번 이상 나타난 리뷰 수


,n_reviews
too_short,14
too_long,1
below_threshold,25
none_cluster,13
below_and_none,12
accepted,0
no_sentence,0


수작업 검토표: /content/drive/MyDrive/shoe_project/results_shoe_absa_20261123_094430/experiment1_followup/03_manual_review.csv
문장별 탈락 내역: /content/drive/MyDrive/shoe_project/results_shoe_absa_20261123_094430/experiment1_followup/03_sentence_trace.csv
입력 가능한 속성: ['디자인', '배송', '착화감', '반품/교환', '내구성', '사진-실물 일치', '기능', '가격/가성비', '사이즈/핏']


#### 수작업 검토 결과 집계 (작업 후 실행해야됨)

In [ ]:
manual = pd.read_csv(
    ANALYSIS_DIR / "03_manual_review.csv", keep_default_na=False
).set_index("review_id", verify_integrity=True)

review_trace_saved = pd.read_csv(
    ANALYSIS_DIR / "03_review_trace.csv", keep_default_na=False
).set_index("review_id", verify_integrity=True)

assert len(manual) == 100 and set(manual.index) == set(review_trace_saved.index)

answers = manual["human_aspects"].str.strip()
if answers.eq("").any():
    raise ValueError(f"아직 작성하지 않은 리뷰가 {answers.eq('').sum()}개 있습니다.")

valid_aspects = {col.rsplit("_", 1)[0] for col in x27_train.columns}

def parse_aspects(value):
    if value == "없음":
        return set()
    result = {part.strip() for part in value.split("|")}
    invalid = result - valid_aspects
    if invalid:
        raise ValueError(f"속성 이름을 확인하세요: {invalid}. 허용 목록: {valid_aspects}")
    return result

audit_rows = []
for review_id, answer in answers.items():
    human = parse_aspects(answer)
    vector = x27_train.loc[review_id]
    system = {col.rsplit("_", 1)[0] for col in vector.index[vector.eq(1)]}

    audit_rows.append({
        "review_id": review_id,
        "sample_group": review_trace_saved.at[review_id, "sample_group"],
        "human_has_aspect": bool(human),
        "exact_aspect_match": human == system,
        "missed_any": bool(human - system),
        "extra_any": bool(system - human),
        "missed_aspects": "|".join(sorted(human - system)),
        "extra_aspects": "|".join(sorted(system - human)),
    })

human_audit = pd.DataFrame(audit_rows)
human_audit.to_csv(
    ANALYSIS_DIR / "03_manual_audit_result.csv", index=False, encoding="utf-8-sig"
)

manual_summary = human_audit.groupby("sample_group").agg(
    n_reviews=("review_id", "size"),
    human_found_aspect=("human_has_aspect", "sum"),
    exact_match=("exact_aspect_match", "sum"),
    reviews_with_missed_aspect=("missed_any", "sum"),
    reviews_with_extra_aspect=("extra_any", "sum"),
)
display(manual_summary)

### 추가실험3 - 같은 실제 별점에서 Mixed vs Single 비교

In [ ]:
matched = pred.loc[
    pred["group"].isin(["mixed", "single"]),
    ["actual_rating", "group", "Random forest"],
].copy()

matched["actual_rating"] = matched["actual_rating"].astype(int)
matched["abs_error"] = (
    matched["actual_rating"] - matched["Random forest"]
).abs()

stars = [1, 2, 3, 4, 5]
groups = ["mixed", "single"]

n_by_star = (
    matched.groupby(["actual_rating", "group"]).size()
    .unstack("group").reindex(index=stars, columns=groups)
)
mae_by_star = (
    matched.groupby(["actual_rating", "group"])["abs_error"].mean()
    .unstack("group").reindex(index=stars, columns=groups)
)

assert n_by_star.notna().all().all() and n_by_star.gt(0).all().all()

pooled_count = n_by_star.sum(axis=1)
common_weight = pooled_count / pooled_count.sum()

star_table = pd.DataFrame({
    "mixed_n": n_by_star["mixed"],
    "single_n": n_by_star["single"],
    "mixed_MAE": mae_by_star["mixed"],
    "single_MAE": mae_by_star["single"],
    "mixed_minus_single": mae_by_star["mixed"] - mae_by_star["single"],
    "common_weight": common_weight,
})

raw_mae = matched.groupby("group")["abs_error"].mean().reindex(groups)
adjusted_mae = mae_by_star.mul(common_weight, axis="index").sum(axis=0)

gap_table = pd.DataFrame({
    "raw_MAE": raw_mae,
    "rating_adjusted_MAE": adjusted_mae,
})
gap_table.loc["mixed_minus_single"] = (
    gap_table.loc["mixed"] - gap_table.loc["single"]
)

star_table.to_csv(ANALYSIS_DIR / "04_MAE_by_actual_rating.csv")
gap_table.to_csv(ANALYSIS_DIR / "04_rating_adjusted_gap.csv")

print("표 1. 같은 실제 별점에서의 비교")
display(star_table.round(4))

print("표 2. 실제 별점 분포를 맞추기 전후의 비교")
display(gap_table.round(4))

표 1. 같은 실제 별점에서의 비교


,mixed_n,single_n,mixed_MAE,single_MAE,mixed_minus_single,common_weight
actual_rating,,,,,,
1,164,1048,1.9255,1.5090,0.4165,0.1295
2,150,651,1.1744,0.9378,0.2366,0.0856
3,235,838,0.5428,0.8253,-0.2825,0.1147
4,257,1284,0.5618,0.7626,-0.2009,0.1647
5,288,4441,1.3338,0.5392,0.7946,0.5055


표 2. 실제 별점 분포를 맞추기 전후의 비교


,raw_MAE,rating_adjusted_MAE
group,,
mixed,1.0494,1.1789
single,0.7574,0.7686
mixed_minus_single,0.2920,0.4104
